In [0]:
# Create widgets for runtime configuration
dbutils.widgets.text("catalog",    "retail_catalog", "Catalog")
dbutils.widgets.text("schema",     "bronze",         "Schema")
dbutils.widgets.text("table_name", "raw_products",   "Table Name")
dbutils.widgets.text("api_url", "", "API URL")
dbutils.widgets.dropdown("load_mode", "incremental", ["full", "incremental"], "Load Mode")

In [0]:
# Read widget values
CATALOG    = dbutils.widgets.get("catalog")
SCHEMA     = dbutils.widgets.get("schema")
TABLE      = dbutils.widgets.get("table_name")
API_URL    = dbutils.widgets.get("api_url")
LOAD_MODE  = dbutils.widgets.get("load_mode")
 
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"
WATERMARK_TABLE = f"{CATALOG}.{SCHEMA}.watermark_log"
 
print(f"Catalog    : {CATALOG}")
print(f"Schema     : {SCHEMA}")
print(f"Table      : {FULL_TABLE}")
print(f"Load Mode  : {LOAD_MODE}")

In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import DataFrame
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_timestamp, when, trim
)

In [0]:
def get_product_schema() -> StructType:
    """Full StructType schema for DummyJSON /products API response."""
    return StructType([
        StructField("id",                   IntegerType(), True),
        StructField("title",                StringType(),  True),
        StructField("description",          StringType(),  True),
        StructField("category",             StringType(),  True),
        StructField("price",                DoubleType(),  True),
        StructField("discountPercentage",   DoubleType(),  True),
        StructField("rating",               DoubleType(),  True),
        StructField("stock",                IntegerType(), True),
        StructField("brand",                StringType(),  True),
        StructField("sku",                  StringType(),  True),
        StructField("weight",               DoubleType(),  True),
        StructField("availabilityStatus",   StringType(),  True),
        StructField("returnPolicy",         StringType(),  True),
        StructField("minimumOrderQuantity", IntegerType(), True),
        StructField("thumbnail",            StringType(),  True),
        StructField("tags",                 ArrayType(StringType()), True),
        StructField("images",               ArrayType(StringType()), True),
        StructField("warrantyInformation",  StringType(),  True),
        StructField("shippingInformation",  StringType(),  True),
        StructField("dimensions", StructType([
            StructField("width",  DoubleType(), True),
            StructField("height", DoubleType(), True),
            StructField("depth",  DoubleType(), True),
        ]), True),
        StructField("meta", StructType([
            StructField("createdAt", StringType(), True),
            StructField("updatedAt", StringType(), True),   # ← WATERMARK FIELD
            StructField("barcode",   StringType(), True),
            StructField("qrCode",    StringType(), True),
        ]), True),
        StructField("reviews", ArrayType(StructType([
            StructField("rating",        DoubleType(), True),
            StructField("comment",       StringType(), True),
            StructField("date",          StringType(), True),
            StructField("reviewerName",  StringType(), True),
            StructField("reviewerEmail", StringType(), True),
        ])), True),
    ])

In [0]:
def create_watermark_table() -> None:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
            table_name      STRING,
            last_updated_at TIMESTAMP,
            load_mode       STRING,
            records_loaded  INT,
            run_at          TIMESTAMP
        )
        USING DELTA
        COMMENT 'Tracks watermark (last loaded timestamp) per table for incremental loads'
    """)
 
 
def get_last_watermark(table_name: str) -> str:
    try:
        result = spark.sql(f"""
            SELECT MAX(last_updated_at) AS last_wm
            FROM {WATERMARK_TABLE}
            WHERE table_name = '{table_name}'
        """).collect()[0]["last_wm"]
        return result.isoformat() if result else "1900-01-01T00:00:00Z"
    except Exception:
        return "1900-01-01T00:00:00Z"

In [0]:
def save_watermark(table_name: str, new_watermark, records_loaded: int, mode: str) -> None:
    wm_schema = StructType([
        StructField("table_name",      StringType(),    True),
        StructField("last_updated_at", TimestampType(), True),
        StructField("load_mode",       StringType(),    True),
        StructField("records_loaded",  IntegerType(),   True),
        StructField("run_at",          TimestampType(), True)
    ])
    wm_data = [(
        table_name,
        new_watermark,
        mode,
        records_loaded,
        datetime.now(timezone.utc)
    )]
    wm_df = spark.createDataFrame(wm_data, schema=wm_schema)
    wm_df.write.format("delta").mode("append").saveAsTable(WATERMARK_TABLE)
    print(f"   Watermark saved → {new_watermark}")

In [0]:
def fetch_all_products(api_url: str) -> list:
    all_products, skip, limit = [], 0, 30
 
    first      = requests.get(f"{api_url}?limit={limit}&skip=0", timeout=30).json()
    total      = first.get("total", 0)
    all_products.extend(first.get("products", []))
    skip += limit
 
    while skip < total:
        page = requests.get(f"{api_url}?limit={limit}&skip={skip}", timeout=30).json()
        all_products.extend(page.get("products", []))
        skip += limit
 
    print(f"  Fetched {len(all_products)} total products from API")
    return all_products
  
def create_bronze_table() -> None:

    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
            -- ── Core Fields ────────────────────────────────────────────
            id                    INT,
            title                 STRING,
            description           STRING,
            category              STRING,
            price                 DOUBLE,
            discountPercentage    DOUBLE,
            rating                DOUBLE,
            stock                 INT,
            brand                 STRING,
            sku                   STRING,
            weight                DOUBLE,
            availabilityStatus    STRING,
            returnPolicy          STRING,
            minimumOrderQuantity  INT,
            warrantyInformation   STRING,
            shippingInformation   STRING,
            thumbnail             STRING,
            tags                  ARRAY<STRING>,
            images                ARRAY<STRING>,
            -- ── Nested Structs ─────────────────────────────────────────
            dimensions            STRUCT<width: DOUBLE, height: DOUBLE, depth: DOUBLE>,
            meta                  STRUCT<
                                    createdAt: STRING,
                                    updatedAt: STRING,
                                    barcode:   STRING,
                                    qrCode:    STRING>,
            reviews               ARRAY<STRUCT<
                                    rating:        DOUBLE,
                                    comment:       STRING,
                                    date:          STRING,
                                    reviewerName:  STRING,
                                    reviewerEmail: STRING>>,
            -- ── Watermark Field (for incremental load) ─────────────────
            _updated_at           TIMESTAMP,   -- parsed from meta.updatedAt
            -- ── Audit Columns ──────────────────────────────────────────
            _ingested_at          TIMESTAMP,
            _load_mode            STRING,
            _source               STRING,
            _layer                STRING
        )
        USING DELTA
        PARTITIONED BY (category)
        COMMENT 'Bronze: Raw retail products — incremental load via _updated_at watermark'
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)
    print(f"   Table {FULL_TABLE} ready")

In [0]:
def run_bronze_load():

    print(f"\n{'='*55}")
    print(f"  Mode : {LOAD_MODE.upper()}")
    print(f"  Table: {FULL_TABLE}")
    print(f"{'='*55}\n")
 
    # ── Fetch from API ────────────────────────────────────────
    products_list = fetch_all_products(API_URL)
    schema        = get_product_schema()
    raw_df        = spark.createDataFrame(products_list, schema=schema)
 
    # ── Parse watermark column from nested meta struct ────────
    raw_df = raw_df.withColumn(
        "_updated_at", to_timestamp(col("meta.updatedAt"))
    )
 
    # ── Incremental filter ────────────────────────────────────
    if LOAD_MODE == "incremental":
        last_wm = get_last_watermark(FULL_TABLE)
        print(f"  Last watermark: {last_wm}")
        raw_df = raw_df.filter(col("_updated_at") > lit(last_wm).cast("timestamp"))
        print(f"  Records after watermark filter: {raw_df.count()}")
    else:
        print(f"  Full load — processing all {raw_df.count()} records")
 
    # ── Add audit columns ─────────────────────────────────────
    final_df = (
        raw_df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_load_mode",   lit(LOAD_MODE))
        .withColumn("_source",      lit(API_URL))
        .withColumn("_layer",       lit("bronze"))
    )
 
    records_count = final_df.count()
 
    if records_count == 0:
        print("   No new records since last watermark — skipping write")
        return
 
    # ── Write to Delta ────────────────────────────────────────
    write_mode = "overwrite" if LOAD_MODE == "full" else "append"
    (
        final_df.write
        .format("delta")
        .mode(write_mode)
        .option("mergeSchema", "true")
        .saveAsTable(FULL_TABLE)
    )
    print(f"   Written {records_count} records (mode={write_mode})")
 
    # ── Save new watermark ────────────────────────────────────
    new_wm = final_df.selectExpr("MAX(_updated_at) AS max_wm").collect()[0]["max_wm"]
    save_watermark(FULL_TABLE, new_wm, records_count, LOAD_MODE)
 
    print(f"\n Bronze load complete!")
    return final_df

In [0]:
create_watermark_table()
create_bronze_table()
run_bronze_load()

In [0]:
%sql
Select * from retail_catalog.bronze.raw_products